In [ ]:
from Utils.import_packages import *

import swifter
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

import string


# Download NLTK data
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

nltk.download('stopwords')
nltk.download('wordnet')

In [3]:
train = pd.read_csv('Data/train_labelled.csv')
test = pd.read_csv('Data/test_labelled.csv')

In [4]:
train

,title,source,topic,company name(s) - cleaned,date,sentiment_label
0,Agilent Technologies Introduces New Version of...,Business Wire,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2014-01-06,1
1,Comcast Corporation Partners with Samsung Elec...,Business Wire,Comcast Corporation (NasdaqGS:CMCSA) (Cable an...,Comcast Corporation,2014-01-06,1
2,Agilent Technologies Introduces ICP-MS and MP-...,Business Wire,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2014-01-06,1
3,Agilent Technologies Inc. Introduces New Exter...,Business Wire,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2014-01-08,1
4,Agilent Technologies Introduces First USB 3.0 ...,Other,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2014-01-09,1
...,...,...,...,...,...,...
187155,"Zoetis Inc., Q3 2021 Earnings Call, Nov 04, 2021",Business Wire; Company Website,Pharmaceuticals,Zoetis Inc.,2021-11-04,1
187156,"Credit Suisse Group AG, 30th Annual Credit Sui...",PR Newswire; Business Wire; GlobeNewswire; Com...,"1Life Healthcare, Inc. (Health Care Services);...","1Life Healthcare, Inc.; 23andMe Holding Co.",2021-11-08,1
187157,Zoetis Inc. Presents at 30th Annual Credit Sui...,PR Newswire; Business Wire; GlobeNewswire; Com...,Pharmaceuticals,Zoetis Inc.,2021-11-09,1
187158,Zoetis Inc. Presents at 2021 HMG Live! Pacific...,GlobeNewswire; Company Website,Pharmaceuticals,Zoetis Inc.,2021-11-18,1


In [16]:
nltk.download()

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


True

In [26]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    
# Preprocessing function
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    # Tokenize text
    tokens = word_tokenize(text.lower(), language='english')
    
    # Get POS tags
    pos_tags = pos_tag(tokens)

    # Remove punctuation and stopwords, and lemmatize
    tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags 
              if word not in stop_words and word not in string.punctuation]
    return ' '.join(tokens)


In [ ]:

train['processed_title'] = train['title'].swifter.apply(preprocess_text)
# val['processed_title'] = val['title'].swifter.apply(preprocess_text)
test['processed_title'] = test['title'].swifter.apply(preprocess_text)


# Prepare data for Naive Bayes
X_train = train['processed_title']
y_train  = train['sentiment_label']

# X_val = val['processed_title']
# y_val  = val['sentiment_label']

X_test = test['processed_title']
y_test  = test['sentiment_label']

# Convert text to numerical features using CountVectorizer
vectorizer = CountVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# Train a Naive Bayes classifier
nb = MultinomialNB()
nb.fit(X_train_vectorized, y_train)

# Make predictions
y_pred = nb.predict(X_test_vectorized)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


Pandas Apply:   0%|          | 0/187160 [00:00<?, ?it/s]